In [1]:
# 接下来在每一种长度下随机选择3个pdb，复制到/home/junjiechen/1_work/250401-Dpepalign/Benchmark-Filter/datasets/PepSet_3per_length目录下，用于后续的测试
import random
import os
import shutil
from collections import defaultdict
from Bio.PDB import PDBParser


with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PepSet/Merged_PDBs/PDB.list', 'r') as file:
    pdbs = [line.strip() for line in file.readlines()]

for pdb in pdbs:
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb, f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PepSet/Merged_PDBs/{pdb}.pdb')
    # 如果链总数大于2，跳过
    chains = list(structure.get_chains())
    if len(chains) > 2:
        continue
    os.system(f'cp /home/junjiechen/1_work/250401-Dpepalign/Benchmark/datasets/PepSet/Merged_PDBs/{pdb}.pdb /home/junjiechen/1_work/250401-Dpepalign/Protenix/PepSet/dimer/{pdb}.pdb')

In [ ]:
# 读取csv文件，将其中sequence列的序列提取出来，保存在一个list中
import os
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import argparse
import pandas as pd
import json
import os

def get_pep_seq(
        pdb_code:str,
        complex_base_dir:str)-> tuple:
    """从complex的pdb文件中提取pro和pep的序列"""
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(pdb_code, f'{complex_base_dir}/{pdb_code}.pdb')
    for model in structure:
        for chain in model:
            if chain.id == 'L':
                pep_sequence = ''
                for residue in chain:
                    if residue.id[0] == ' ':
                        one_letter_resname = seq1(residue.get_resname())
                        pep_sequence += one_letter_resname
            else:
                pro_sequence = ''
                for residue in chain:
                    if residue.id[0] == ' ':
                        one_letter_resname = seq1(residue.get_resname())
                        pro_sequence += one_letter_resname
    return pro_sequence, pep_sequence




json_temple_path = "../template.json"

#根据template.json的内容以及fasta文件，编写json文件，将fasta中的pro序列替换template.json中的pro_sequence，将fasta中的pep序列替换template.json中的pep_sequence，保存为新的json文件


with open ('./dimer/PDB.list', 'r') as f:
    pdbs = f.read().splitlines()

    jobs = []

for pdb in pdbs:
    with open(json_temple_path, 'r') as file:
        tmpl = json.load(file)

    if pdb in os.listdir("./MSA_pro_all_PepSet_dimer"):
        msa_pro_path = f'./MSA_pro_all_PepSet_dimer/{pdb}'
        pairing_db = 'uniref100'
    else:
        msa_pro_path = ""
        pairing_db = ""

        # 编写多肽的msa文件
    complex_base_dir = './dimer'
    seq_pro, seq_pep = get_pep_seq(pdb, complex_base_dir)

    msa_pep_path = f'MSA_pep/{pdb}'
    # os.makedirs(msa_pep_path, exist_ok=True)
    # with open(f'{msa_pep_path}/pairing.a3m', 'w') as f:
    #     f.write('>query\n')
    #     f.write(seq_pep + '\n')
    # os.system(f'cp {msa_pep_path}/pairing.a3m {msa_pep_path}/non_pairing.a3m')
    
    job = json.loads(json.dumps(tmpl[0]))

    # 如果msa存在，则替换msa路径和pairing_db，如果不存在，则不添加msa路径和pairing_db
    if msa_pro_path:
        job['sequences'][0]['proteinChain']['msa']['precomputed_msa_dir'] = msa_pro_path
        job['sequences'][0]['proteinChain']['msa']['pairing_db'] = pairing_db
        job['sequences'][1]['proteinChain']['msa']['precomputed_msa_dir'] = msa_pep_path
        job['sequences'][1]['proteinChain']['msa']['pairing_db'] = pairing_db
    else:
        job['sequences'][0]['proteinChain'].pop('msa', None)
        job['sequences'][1]['proteinChain'].pop('msa', None)
    job['name'] = pdb
    job['sequences'][0]['proteinChain']['sequence'] = seq_pro
    job['sequences'][1]['proteinChain']['sequence'] = seq_pep

    jobs.append(job.copy())


with open(f'job2.json', 'w') as f:
    f.write(json.dumps(jobs, indent=4))

In [3]:
# 检查job.json中有多少个任务，包含msa字段的有几个任务
with open('job2.json', 'r') as f:
    jobs = json.load(f)
    total_jobs = len(jobs)
    jobs_with_msa = sum(1 for job in jobs if 'msa' in job['sequences'][0]['proteinChain'] and 'msa' in job['sequences'][1]['proteinChain'])
    print(f'Total jobs: {total_jobs}')
    print(f'Jobs with MSA: {jobs_with_msa}')



Total jobs: 170
Jobs with MSA: 170
